# Modeling: predicting long subway delays (≥ 5 min)

Input: `data/processed/delays_clean.csv`, produced by `01_eda.ipynb`.

In [1]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.base import clone

df = pd.read_csv("../data/processed/delays_clean.csv", parse_dates=["datetime"])
print(df.shape)
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

(45454, 13)
Date range: 2025-01-01 00:27:00 to 2026-08-31 23:59:00


## Time-based split and baseline

In [2]:
# Time-based split: train on the earliest 80% of incidents, test on the latest 20%.
# A random split would let the model "see the future" during training.
df = df.sort_values("datetime").reset_index(drop=True)
cutoff = df["datetime"].quantile(0.8)

train = df[df["datetime"] < cutoff]
test = df[df["datetime"] >= cutoff]

for name, part in [("Train", train), ("Test", test)]:
    print(f"{name}: {len(part):,} incidents, "
          f"{part['datetime'].min():%Y-%m-%d} to {part['datetime'].max():%Y-%m-%d}, "
          f"long-delay share {part['long_delay'].mean():.1%}")

Train: 36,363 incidents, 2025-01-01 to 2026-05-12, long-delay share 18.9%
Test: 9,091 incidents, 2026-05-12 to 2026-08-31, long-delay share 14.3%


In [3]:
FEATURES = [
    "station_clean", "line_clean", "code_feature", "cause_group",
    "time_period", "hour", "weekday", "month", "is_weekend", "is_rush_hour",
]
TARGET = "long_delay"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

In [4]:
# Baseline: always predict the most common class (not a long delay)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print(confusion_matrix(y_test, y_pred))

Accuracy:  0.857
Precision: 0.000
Recall:    0.000
F1:        0.000
[[7795    0]
 [1296    0]]


Split and baseline:
- Data covers Jan 2025 to Aug 2026. Train: Jan 2025 - May 12, 2026 (36,363 incidents, 18.9% long);
  test: May 12 - Aug 31, 2026 (9,091 incidents, 14.3% long).
- The long-delay share is lower in the test period (possible seasonality or trend), so the test set
  is not distributed exactly like the training set.
- Majority-class baseline: accuracy 0.857 but precision, recall and F1 are all 0. Accuracy is
  misleading here, so models are compared mainly on F1 and recall.

## Why is the test period different?

In [5]:
# Long-delay share by month: is the lower share in the test period seasonal or a trend?
monthly = df.groupby(df["datetime"].dt.to_period("M"))["long_delay"].agg(["mean", "count"])
monthly["mean"] = monthly["mean"].round(3)
monthly

,mean,count
datetime,,
2025-01,0.218,2106
2025-02,0.203,2274
2025-03,0.232,1986
2025-04,0.198,2117
2025-05,0.184,2174
2025-06,0.209,1939
2025-07,0.176,2121
2025-08,0.141,2718
2025-09,0.168,1811


Monthly long-delay share shows both seasonality and a shift in 2026:
- Seasonal: higher in winter (20-23% in Jan-Mar 2025, 21.8% in Jan 2026), lowest in late summer (14.1% in Aug 2025).
- Shift: summer 2026 is lower than summer 2025 (e.g. June 14.3% vs 20.9%), while the number of
  recorded incidents per month is higher in 2026. One possible explanation is that more minor
  incidents are being recorded; the data alone cannot confirm this.
- Since the model learns from 2025 patterns, it may over-predict long delays in the 2026 test period.

## Logistic regression

In [6]:
results = []

def evaluate(name, y_true, y_pred):
    """Print and store the main metrics for one model."""
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }
    results.append(row)
    print(pd.Series(row).drop("model").round(3).to_string())
    print(confusion_matrix(y_true, y_pred))

evaluate("Majority baseline", y_test, baseline.predict(X_test))

accuracy     0.857
precision      0.0
recall         0.0
f1             0.0
[[7795    0]
 [1296    0]]


In [7]:
# Categorical features are one-hot encoded; the two 0/1 flags are used as they are.
# Hour is left out: time_period already captures the non-linear hourly pattern.
CATEGORICAL = ["station_clean", "line_clean", "code_feature", "cause_group",
               "time_period", "weekday", "month"]
BINARY = ["is_weekend", "is_rush_hour"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("bin", "passthrough", BINARY),
])

for name, weights in [("Logistic regression", None),
                      ("Logistic regression (balanced)", "balanced")]:
    model = Pipeline([
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(max_iter=1000, class_weight=weights)),
    ])
    model.fit(X_train, y_train)
    print(f"--- {name} ---")
    evaluate(name, y_test, model.predict(X_test))
    print()
lr_balanced = model  # keep the balanced logistic regression for inspection below

--- Logistic regression ---
accuracy     0.862
precision    0.526
recall         0.3
f1           0.382
[[7444  351]
 [ 907  389]]

--- Logistic regression (balanced) ---
accuracy     0.709
precision    0.305
recall       0.814
f1           0.444
[[5393 2402]
 [ 241 1055]]



Logistic regression results (test period May-Aug 2026):
- Both versions clearly beat the baseline (F1 0.38 and 0.44 vs 0), so the features carry real signal.
- Default weights: precision 0.53, recall 0.30. It is conservative and misses most long delays.
- Balanced class weights: recall 0.81, precision 0.31, best F1 (0.44). It catches most long delays
  at the cost of many false alarms.
- The choice depends on the use case: an early-warning tool would favour recall.
- Accuracy is lowest for the model with the best F1, confirming it is not a useful metric here.

## Inspecting the logistic regression

In [8]:
# Look inside the pipeline: how many columns does one-hot encoding create?
feature_names = lr_balanced.named_steps["preprocess"].get_feature_names_out()
print("Number of columns after one-hot encoding:", len(feature_names))
print(feature_names[:10])

Number of columns after one-hot encoding: 184
['cat__station_clean_BATHURST STATION' 'cat__station_clean_BAY STATION'
 'cat__station_clean_BAYVIEW STATION'
 'cat__station_clean_BESSARION STATION' 'cat__station_clean_BETWEEN'
 'cat__station_clean_BLOOR STATION' 'cat__station_clean_BROADVIEW STATION'
 'cat__station_clean_CASTLE FRANK STATION'
 'cat__station_clean_CEDARVALE YU STATION'
 'cat__station_clean_CHESTER STATION']


In [9]:
# Logistic regression coefficients: positive = raises the chance of a long delay,
# negative = lowers it
coefs = pd.Series(lr_balanced.named_steps["classifier"].coef_[0], index=feature_names).sort_values()
print("Most positive:")
print(coefs.tail(10).round(2))
print("\nMost negative:")
print(coefs.head(10).round(2))

Most positive:
cat__code_feature_PUTO        1.70
cat__code_feature_TUCC        1.77
cat__code_feature_MUSAN       1.82
cat__code_feature_PUTWZ       1.97
cat__code_feature_PUTR        1.99
cat__code_feature_MU_OTHER    2.00
cat__code_feature_MUIR        2.03
cat__code_feature_MUDD        2.29
cat__code_feature_PUSSW       2.39
cat__code_feature_MUI         2.71
dtype: float64

Most negative:
cat__code_feature_TUNCA        -5.24
cat__code_feature_PUMEL        -5.21
cat__code_feature_MUNCA        -4.55
cat__code_feature_PUMST        -4.41
cat__code_feature_MUCP         -3.01
cat__code_feature_EUSC         -2.81
cat__station_clean_LINE-WIDE   -2.71
cat__code_feature_MUIS         -2.67
cat__code_feature_MUIRS        -2.08
cat__line_clean_MULTI          -2.08
dtype: float64


In [10]:
# Surprising result: are line-wide and multi-line incidents really less likely to be long?
for col, value in [("station_clean", "LINE-WIDE"), ("line_clean", "MULTI")]:
    subset = df[df[col] == value]
    print(f"{col} = {value}: {len(subset)} incidents, "
          f"long-delay share {subset['long_delay'].mean():.1%}, "
          f"zero-minute share {(subset['Min Delay'] == 0).mean():.1%}")

station_clean = LINE-WIDE: 1195 incidents, long-delay share 0.3%, zero-minute share 99.6%
line_clean = MULTI: 597 incidents, long-delay share 0.0%, zero-minute share 100.0%


Inspecting the logistic regression (balanced):
- One-hot encoding turns the 9 features into 184 columns.
- The largest positive coefficients are the codes with the highest long-delay share in the EDA
  (e.g. injured customer on train `MUI`, track switch failure `PUSSW`, rail problems `PUTR`).
- The largest negative coefficients are codes that are not train delays at all
  (no collector, escalator/stairway incidents, planned closures).
- Surprisingly, `LINE-WIDE` locations and `MULTI` lines also have strong negative coefficients.
  Checking the data confirms it: 99.6% of line-wide and 100% of multi-line records have a
  0-minute delay, so they are mostly notices rather than service disruptions.

## Random forest

In [11]:
# Same preprocessing as the logistic regression above; only the classifier changes
for name, weights in [("Random Forest", None),
                      ("Random Forest (balanced)", "balanced")]:
    model = Pipeline([
        ("preprocess", preprocess),
        ("classifier", RandomForestClassifier(
            n_estimators=300, class_weight=weights, min_samples_leaf=20,
            random_state=42, n_jobs=-1,
        )),
    ])
    model.fit(X_train, y_train)
    print(f"--- {name} ---")
    evaluate(name, y_test, model.predict(X_test))
    print()

--- Random Forest ---
accuracy     0.863
precision    0.584
recall        0.14
f1           0.225
[[7666  129]
 [1115  181]]

--- Random Forest (balanced) ---
accuracy     0.695
precision    0.294
recall       0.812
f1           0.431
[[5266 2529]
 [ 244 1052]]



Random forest results:
- With balanced class weights: precision 0.29, recall 0.81, F1 0.43, slightly below the balanced
  logistic regression (F1 0.44). Without weights it is very conservative (recall 0.14).
- The more flexible model does not help here. A likely reason is that most of the signal comes
  from the cause code alone, which logistic regression already captures, so there are few
  interactions left for the trees to exploit.

## Threshold tuning

In [12]:
# Hold out the last 20% of the training period as a validation set.
# The threshold is chosen on this set, so the test set is only used once at the end.
val_cutoff = train["datetime"].quantile(0.8)
inner = train[train["datetime"] < val_cutoff]
val = train[train["datetime"] >= val_cutoff]

for name, part in [("Inner train", inner), ("Validation", val)]:
    print(f"{name}: {len(part):,} incidents, "
          f"{part['datetime'].min():%Y-%m-%d} to {part['datetime'].max():%Y-%m-%d}, "
          f"long-delay share {part['long_delay'].mean():.1%}")

Inner train: 29,090 incidents, 2025-01-01 to 2026-02-12, long-delay share 19.4%
Validation: 7,273 incidents, 2026-02-12 to 2026-05-12, long-delay share 16.7%


In [13]:
# Train a standard (unweighted) logistic regression on the inner training set,
# then try different probability thresholds on the validation set.
lr = Pipeline([
    ("preprocess", clone(preprocess)),
    ("classifier", LogisticRegression(max_iter=1000)),
])
lr.fit(inner[FEATURES], inner[TARGET])
val_proba = lr.predict_proba(val[FEATURES])[:, 1]  # probability of a long delay

rows = []
for t in np.arange(0.05, 0.80, 0.05):
    pred = (val_proba >= t).astype(int)
    rows.append({
        "threshold": round(t, 2),
        "precision": precision_score(val[TARGET], pred, zero_division=0),
        "recall": recall_score(val[TARGET], pred),
        "f1": f1_score(val[TARGET], pred),
    })

threshold_table = pd.DataFrame(rows).set_index("threshold").round(3)
best_threshold = threshold_table["f1"].idxmax()
print("Best threshold on validation:", best_threshold)
threshold_table

Best threshold on validation: 0.3


,precision,recall,f1
threshold,,,
0.05,0.233,0.979,0.376
0.10,0.246,0.954,0.391
0.15,0.276,0.901,0.422
0.20,0.307,0.800,0.444
0.25,0.358,0.719,0.478
0.30,0.415,0.634,0.501
0.35,0.454,0.545,0.495
0.40,0.502,0.474,0.488
0.45,0.549,0.399,0.462


In [14]:
# Retrain on the full training set, then apply the chosen threshold to the test set
lr.fit(X_train, y_train)
test_proba = lr.predict_proba(X_test)[:, 1]
evaluate(f"Logistic regression (threshold {best_threshold})",
         y_test, (test_proba >= best_threshold).astype(int))

accuracy      0.81
precision    0.393
recall       0.608
f1           0.477
[[6578 1217]
 [ 508  788]]


Threshold tuning:
- A standard logistic regression was trained on Jan 2025 - Feb 2026, and the probability threshold
  was chosen on a validation period (Feb 12 - May 12, 2026). F1 peaks at 0.30 (0.50), with a
  fairly flat region between 0.25 and 0.40, so the choice is not sensitive.
- The default threshold of 0.5 is too conservative because long delays are only about 18% of incidents.
- On the test set, threshold 0.30 gives the best F1 so far (0.477): precision 0.39, recall 0.61.
  Compared with balanced class weights, it halves the number of false alarms (2,402 → 1,217)
  at the cost of lower recall.
- Validation F1 (0.50) is slightly above test F1 (0.48), consistent with the lower long-delay
  share in the test period.

## Summary

In [15]:
# Final comparison of all models on the test set (May-Aug 2026)
pd.DataFrame(results).set_index("model").round(3)

,accuracy,precision,recall,f1
model,,,,
Majority baseline,0.857,0.000,0.000,0.000
Logistic regression,0.862,0.526,0.300,0.382
Logistic regression (balanced),0.709,0.305,0.814,0.444
Random Forest,0.863,0.584,0.140,0.225
Random Forest (balanced),0.695,0.294,0.812,0.431
Logistic regression (threshold 0.3),0.810,0.393,0.608,0.477


- Best F1: logistic regression with a tuned threshold of 0.30 (0.477), ahead of balanced
  class weights (0.444) with half as many false alarms.
- Balanced class weights trade precision for recall: both balanced models catch about 81% of
  long delays, but only about 30% of their alarms are correct.